<a href="https://colab.research.google.com/github/saridango/csci164-sp25/blob/main/Copy_of_AI24Ch3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Summary of Additions

-- Generated 15 random problems in the 3x3 and 15 random problems in the 4x4  puzzles by generating:

* 3: Random Walk 5 steps

* 3: Random Walk 10 steps

* 3: Random Walk 20 steps

* 3: Random Walk 40 steps

* 3: Random Walk 80 steps

-- Evaluated results and explained what they can indicate about search as a tool for AI, especially as related to increase complexity in problem size.


In [8]:
import random
import heapq

# Tile Sliding Domain: Initial State Space

In [60]:
StateDimension = 4  # Change from 3 to 4
InitialState = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 13, 14, 15]
GoalState = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite=dict([('u','d'),('d','u'),('l','r'),('r','l'), (None, None)])

In [10]:
def Result(state, action):
  i = state.index(0)
  newState = list(state)
  row,col=i//StateDimension, i % StateDimension
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return newState
  if action=='u':
    l,r = row*StateDimension+col, (row-1)*StateDimension+col
  elif action=='d':
    l,r = row*StateDimension+col, (row+1)*StateDimension+col
  elif action=='l':
    l,r = row*StateDimension+col, row*StateDimension+col-1
  elif action=='r' :
    l,r = row*StateDimension+col, row*StateDimension+col+1
  newState[l], newState[r] = newState[r], newState[l]
  return newState

def PrintState(s):
  for i in range(0,len(s),StateDimension):
    print(s[i:i+StateDimension])

def LegalMove(state, action):
  i = state.index(0)
  row,col=i//StateDimension, i % StateDimension
  newState = state.copy()
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return False
  return True


In [65]:
def SingleTileManhattanDistance(tile, left, right):
  leftIndex = left.index(tile)
  rightIndex = right.index(tile)
  return (abs(leftIndex//StateDimension-rightIndex//StateDimension) +
          abs(leftIndex%StateDimension-rightIndex%StateDimension))

def ManhattanDistance(left, right):
  distances = [SingleTileManhattanDistance(tile, left, right)
     for tile in range(1, StateDimension**2)]
  ### print ("Distances= ", distances)
  return sum(distances)


In [66]:
def OutOfPlace(left, right):
  distances = [left[i]!=right[i] and right[i] != 0
     for i in range(StateDimension**2)]
  return sum(distances)

In [67]:
PrintState(InitialState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[0, 13, 14, 15]


In [68]:
PrintState(GoalState)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [69]:
print("ManhattanDistance=  ", ManhattanDistance(InitialState, GoalState))
print("OutOfPlace= ", OutOfPlace(InitialState, GoalState))


ManhattanDistance=   3
OutOfPlace=  3


In [70]:
PrintState(InitialState)
print()
state1 = Result(InitialState, 'u')
PrintState(state1)
print()
state1 = Result(state1, 'r')
PrintState(state1)

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[0, 13, 14, 15]

[1, 2, 3, 4]
[5, 6, 7, 8]
[0, 10, 11, 12]
[9, 13, 14, 15]

[1, 2, 3, 4]
[5, 6, 7, 8]
[10, 0, 11, 12]
[9, 13, 14, 15]


# Random Walk

Take some random moves from a state and return the new state and the sequence of moves.

Do not include moves undoing last move, or having no effect.

In [71]:
def RandomWalk(state, steps):
  actionSequence = []
  actionLast = None
  for i in range(steps):
    action = None
    while action==None:
      action = random.choice(Actions(state))
      action = action if (LegalMove(state, action)
          and action!= Opposite[actionLast]) else None
    actionLast = action
    state = Result(state, action)
    actionSequence.append(action)
  return state, actionSequence



In [72]:
state1, sol = RandomWalk(InitialState, 150)
PrintState(state1)
print (ManhattanDistance(state1, GoalState), sol)

state1, sol = RandomWalk(InitialState, 5)
PrintState(InitialState)
print (sol)
PrintState(state1)

[7, 1, 10, 5]
[15, 6, 2, 4]
[12, 13, 8, 14]
[9, 11, 0, 3]
35 ['u', 'r', 'r', 'u', 'r', 'u', 'l', 'l', 'd', 'l', 'u', 'r', 'r', 'd', 'l', 'l', 'd', 'r', 'r', 'r', 'd', 'l', 'u', 'r', 'd', 'l', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'u', 'l', 'd', 'l', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'l', 'u', 'r', 'd', 'l', 'd', 'd', 'r', 'r', 'u', 'u', 'r', 'u', 'l', 'd', 'l', 'l', 'd', 'd', 'r', 'r', 'r', 'u', 'l', 'l', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'd', 'l', 'l', 'u', 'r', 'r', 'd', 'l', 'l', 'u', 'r', 'd', 'r', 'r', 'u', 'l', 'd', 'l', 'l', 'u', 'r', 'r', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'l', 'l', 'u', 'l', 'u', 'r', 'r', 'd', 'r', 'u', 'l', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'l', 'u', 'l', 'l', 'd', 'r', 'd', 'l', 'd', 'r', 'u', 'r', 'd']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[0, 13, 14, 15]
['u', 'u', 'r', 'd', 'r']
[1, 2, 3, 4]
[6, 10, 7, 8]
[5, 11, 0, 12]
[9, 13, 14, 15]


In [73]:
def ApplyMoves(actions, state):
  for action in actions:
    state = Result(state, action)
  return state

In [74]:
PrintState(InitialState)
print(['r','r'])
PrintState(ApplyMoves(['r','r'],InitialState))

[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[0, 13, 14, 15]
['r', 'r']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 0, 15]


In [75]:
def ReverseMoves(actions):
  ret = [Opposite[a] for a in actions]
  ret.reverse()
  return ret

In [76]:
state1, sol = RandomWalk(GoalState, 5)
PrintState(state1)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))


[1, 2, 3, 4]
[5, 10, 6, 8]
[9, 0, 7, 11]
[13, 14, 15, 12]
['u', 'l', 'u', 'l', 'd']
['u', 'r', 'd', 'r', 'd']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


## Problem Class

INITIAL = InitialState  
IsGoal = Goal Test  
Actions = Actions List  
Result = Action Behavior  
ActionCost = Action Cost  

In [77]:
class Problem(object): pass

## Node

In [78]:
class Node(object):
  def __init__(self, state, parent=None, action=None, path_cost=0 ):
    self.State=state
    self.Parent=parent
    self.Action=action
    self.PathCost = path_cost

  def __str__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __repr__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __lt__(self, other):
    return self.PathCost < other.PathCost;

## Expand

In [79]:
def Expand(problem, node):
  ret = []
  s = node.State
  for action in problem.Actions(s):
    sPrime = problem.Result(s, action)
    cost =node.PathCost + problem.ActionCost(s,action,sPrime)
    ret.append(Node(sPrime, node, action, cost))
  return ret


## Breadth-First Search

In [80]:
def BreadthFirstSearch(problem):
  node = Node(tuple(problem.INITIAL))
  if problem.IsGoal(node.State):
    return node, 0
  Frontier = []
  Frontier.append(node)
  reached = set()
  reached.add(tuple(problem.INITIAL))
  nodesExpanded = 0
  while (Frontier):
    ### print([str(n) for n in Frontier])
    node = Frontier.pop(0)
    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      ### print (s, "IsGoal=", problem.IsGoal(s))
      if problem.IsGoal(s):
        return child, nodesExpanded
      if s not in reached:
        reached.add(s)
        Frontier.append(child)
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

## Best-First Search

In [81]:
def BestFirstSearch(problem, f):
  node = Node(tuple(problem.INITIAL))
  Frontier = []
  heapq.heappush(Frontier,(f(node), node))
  reached = {}
  reached[tuple(problem.INITIAL)]=node
  nodesExpanded = 0
  while (Frontier):
    ##print([(x, str(n)) for (x,n) in Frontier])
    fValue, node = heapq.heappop(Frontier)
    ##print (node.State, "IsGoal=", problem.IsGoal(tuple(node.State)))
    if problem.IsGoal(tuple(node.State)):
      return node, nodesExpanded    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      if s not in reached or child.PathCost < reached[s].PathCost:
        reached[s] = child
        heapq.heappush(Frontier, (f(child), child))
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

## Problem 1

In [87]:
TileSliding = Problem()
TileSliding.INITIAL = InitialState  # InitialState is already defined as a 4x4 puzzle
TileSliding.IsGoal = lambda s: s == (1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0)
TileSliding.Actions = Actions
TileSliding.Result = Result
TileSliding.ActionCost = lambda s, a, sPrime: 1
print(TileSliding.IsGoal((1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0)))
print(Node(InitialState))
print(1 + TileSliding.ActionCost(1, 2, 3))

True
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 0, 13, 14, 15], <none>
2


In [88]:
TileSliding.INITIAL = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
ret, cost = BreadthFirstSearch(TileSliding)
print (ret, cost)

(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0), <none> 0


In [85]:
def Solution(node):
  if node.Parent==None:
    return []
  return Solution(node.Parent) + [node.Action]


In [89]:
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))

[]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]


In [91]:
TileSliding.INITIAL = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
ret, cost = BreadthFirstSearch(TileSliding)
print (ret, cost)

(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0), <none> 0


In [92]:
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))

[]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]


In [94]:
UniformCostF = lambda n: n.PathCost
AStarF = lambda n: n.PathCost+ManhattanDistance(n.State, GoalState)
TileSliding.INITIAL = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
ret, cost = BestFirstSearch(TileSliding, UniformCostF)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Nodes Expanded=", cost)

(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0), <none>
[]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Nodes Expanded= 0


# Problem 2

In [109]:
state1, sol = RandomWalk(GoalState, 80)
PrintState(state1)
print (sol)
print(ReverseMoves(sol))
PrintState (ApplyMoves(ReverseMoves(sol), state1))

[2, 5, 11, 8]
[1, 12, 9, 3]
[4, 6, 7, 14]
[10, 0, 15, 13]
['l', 'l', 'u', 'u', 'r', 'r', 'u', 'l', 'd', 'l', 'l', 'd', 'd', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'r', 'r', 'u', 'l', 'd', 'l', 'd', 'l', 'u', 'r', 'r', 'r', 'u', 'l', 'd', 'd', 'l', 'u', 'r', 'u', 'r', 'd', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'l', 'u', 'u', 'l', 'd', 'l', 'd', 'd', 'r', 'u', 'r', 'u', 'r', 'd', 'd', 'l', 'u', 'u', 'u', 'l', 'd', 'd', 'd', 'r', 'u', 'l', 'd', 'l', 'u', 'r', 'd']
['u', 'l', 'd', 'r', 'u', 'r', 'd', 'l', 'u', 'u', 'u', 'r', 'd', 'd', 'd', 'r', 'u', 'u', 'l', 'd', 'l', 'd', 'l', 'u', 'u', 'r', 'u', 'r', 'd', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'u', 'u', 'l', 'd', 'l', 'd', 'r', 'u', 'u', 'r', 'd', 'l', 'l', 'l', 'd', 'r', 'u', 'r', 'u', 'r', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'd', 'd', 'l', 'u', 'u', 'r', 'r', 'u', 'r', 'd', 'l', 'l', 'd', 'd', 'r', 'r']
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


In [110]:
TileSliding.INITIAL = state1
ret, cost = BreadthFirstSearch(TileSliding)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Length of solution: ", len(sol))
print ("Nodes Expanded=", cost)

KeyboardInterrupt: 

In [108]:
UniformCostF = lambda n: n.PathCost
TileSliding.INITIAL = state1
ret, cost = BestFirstSearch(TileSliding, UniformCostF)
print (ret)
sol = Solution(ret)
print (sol)
print (TileSliding.INITIAL)
print (ApplyMoves(sol, TileSliding.INITIAL))
print ("Length of solution: ", len(sol))
print ("Nodes Expanded=", cost)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0], d
['l', 'u', 'r', 'u', 'u', 'r', 'r', 'd', 'd', 'd']
[1, 6, 2, 3, 5, 10, 7, 4, 13, 9, 11, 8, 14, 0, 15, 12]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3546


# Problem List

In [172]:
findNum = 10
randomWalkDistance = 5
problemList = []
for i in range(3):
  state1, sol = RandomWalk(GoalState, 10)
  problemList.append(state1)
print (problemList)

[[0, 1, 2, 7, 5, 6, 4, 3, 9, 10, 11, 8, 13, 14, 15, 12], [2, 5, 0, 4, 1, 7, 3, 8, 9, 6, 10, 11, 13, 14, 15, 12], [1, 2, 3, 4, 9, 0, 6, 7, 13, 5, 11, 8, 14, 10, 15, 12]]


### Breadth First Search w/ Test Problems

In [151]:
Solutions = []
for s in problemList:
  TileSliding.INITIAL = s
  ret, cost = BreadthFirstSearch(TileSliding)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)


['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2214
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1531
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2250
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 2214), ('2534101179610813141512', 'uldrdrurdd', 1531), ('1273564891011121301415', 'rruulurddd', 2250)]


#Results:

# 4x4 5 Steps
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2250
-----------------------
['r', 'd', 'd', 'd', 'l', 'l', 'u', 'r', 'r', 'd']
-----------------------
[1, 2, 0, 3, 5, 6, 7, 4, 9, 14, 10, 8, 13, 15, 12, 11]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2146
-----------------------
['d', 'r', 'u', 'l', 'u', 'r', 'r', 'd', 'd', 'd']
-----------------------
[1, 6, 2, 3, 5, 0, 11, 4, 9, 7, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2056
-----------------------
-------
[('1273564891011121301415', 'rruulurddd', 2250), ('1203567491410813151211', 'rdddllurrd', 2146), ('1623501149710813141512', 'drulurrddd', 2056)]

#4x4 10 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1724
-----------------------
['d', 'r', 'u', 'u', 'u', 'l', 'd', 'r', 'd', 'd']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2156
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2043
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 1724), ('1247563891001213141115', 'druuuldrdd', 2156), ('2304110685971113141512', 'llddrurdrd', 2043)]

#4x4 20 Steps
['d', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 13, 11, 8, 9, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1420
-----------------------
['r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'd']
-----------------------
[1, 3, 4, 8, 5, 2, 7, 11, 9, 6, 0, 10, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2768
-----------------------
['d', 'r', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 8, 11, 0, 10, 7, 12, 9, 13, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1715
-----------------------
-------
[('1234567010131189141512', 'ddlluldrrr', 1420), ('1348527119601013141512', 'ruullddrrd', 2768), ('1234568110107129131415', 'drrruuldrd', 1715)]

#4x4 40 Steps
['l', 'u', 'l', 'u', 'l', 'd', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 6, 10, 7, 8, 5, 11, 14, 12, 9, 13, 15, 0]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1479
-----------------------
['l', 'u', 'u', 'u', 'r', 'r', 'd', 'r', 'd', 'd']
-----------------------
[5, 1, 2, 4, 9, 6, 3, 7, 13, 10, 11, 8, 14, 0, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1767
-----------------------
['l', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 6, 7, 8, 5, 9, 15, 11, 13, 10, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2050
-----------------------
-------
[('1234610785111412913150', 'lululddrrr', 1479), ('5124963713101181401512', 'luuurrdrdd', 1767), ('2304167859151113101412', 'llddrdrurd', 2050)]

#4x4 80 Steps
['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2214
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 1531
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2250
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 2214), ('2534101179610813141512', 'uldrdrurdd', 1531), ('1273564891011121301415', 'rruulurddd', 2250)]

### Uniform Cost Search w/ Test Problems

In [152]:
UniformCostF = lambda n: n.PathCost

Solutions = []
for s in problemList:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, UniformCostF)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 4782
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 4705
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3948
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 4782), ('2534101179610813141512', 'uldrdrurdd', 4705), ('1273564891011121301415', 'rruulurddd', 3948)]


#Results:
#4x4 5 Steps
['d', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 13, 11, 8, 9, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2716
-----------------------
['r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'd']
-----------------------
[1, 3, 4, 8, 5, 2, 7, 11, 9, 6, 0, 10, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 5710
-----------------------
['d', 'r', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 8, 11, 0, 10, 7, 12, 9, 13, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3509
-----------------------
-------
[('1234567010131189141512', 'ddlluldrrr', 2716), ('1348527119601013141512', 'ruullddrrd', 5710), ('1234568110107129131415', 'drrruuldrd', 3509)]

#4x4 10 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 4638
-----------------------
['r', 'u', 'u', 'l', 'd', 'r', 'd', 'l', 'd', 'r']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 4442
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3510
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 4638), ('1247563891001213141115', 'ruuldrdldr', 4442), ('2304110685971113141512', 'llddrurdrd', 3510)]

#4x4 20 Steps
['d', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 13, 11, 8, 9, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 2716
-----------------------
['r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'd']
-----------------------
[1, 3, 4, 8, 5, 2, 7, 11, 9, 6, 0, 10, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 5710
-----------------------
['d', 'r', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 8, 11, 0, 10, 7, 12, 9, 13, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3509
-----------------------
-------
[('1234567010131189141512', 'ddlluldrrr', 2716), ('1348527119601013141512', 'ruullddrrd', 5710), ('1234568110107129131415', 'drrruuldrd', 3509)]

#4x4 40 Steps
['l', 'u', 'l', 'u', 'l', 'd', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 6, 10, 7, 8, 5, 11, 14, 12, 9, 13, 15, 0]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3682
-----------------------
['l', 'u', 'u', 'u', 'r', 'r', 'd', 'r', 'd', 'd']
-----------------------
[5, 1, 2, 4, 9, 6, 3, 7, 13, 10, 11, 8, 14, 0, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3411
-----------------------
['l', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 6, 7, 8, 5, 9, 15, 11, 13, 10, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3239
-----------------------
-------
[('1234610785111412913150', 'lululddrrr', 3682), ('5124963713101181401512', 'luuurrdrdd', 3411), ('2304167859151113101412', 'llddrdrurd', 3239)]

#4x4 80 Steps
['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 4782
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 4705
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 3948
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 4782), ('2534101179610813141512', 'uldrdrurdd', 4705), ('1273564891011121301415', 'rruulurddd', 3948)]

### AStar using ManhattanDistance w/ Test Problems

In [153]:
AStarFb = lambda n: n.PathCost + Manhattan(n.State, GoalState)

Solutions = []
for s in problemList:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarF)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 21
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 16
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 23
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 21), ('2534101179610813141512', 'uldrdrurdd', 16), ('1273564891011121301415', 'rruulurddd', 23)]


#Results
#4x4 5 Steps
['d', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 13, 11, 8, 9, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 23
-----------------------
['r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'd']
-----------------------
[1, 3, 4, 8, 5, 2, 7, 11, 9, 6, 0, 10, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
['d', 'r', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 8, 11, 0, 10, 7, 12, 9, 13, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 14
-----------------------
-------
[('1234567010131189141512', 'ddlluldrrr', 23), ('1348527119601013141512', 'ruullddrrd', 11), ('1234568110107129131415', 'drrruuldrd', 14)]

#4x4 10 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['d', 'r', 'u', 'u', 'u', 'l', 'd', 'r', 'd', 'd']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 35
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 10), ('1247563891001213141115', 'druuuldrdd', 35), ('2304110685971113141512', 'llddrurdrd', 10)]
#4x4 20 Steps
['d', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 13, 11, 8, 9, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 23
-----------------------
['r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'd']
-----------------------
[1, 3, 4, 8, 5, 2, 7, 11, 9, 6, 0, 10, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
['d', 'r', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 8, 11, 0, 10, 7, 12, 9, 13, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 14
-----------------------
-------
[('1234567010131189141512', 'ddlluldrrr', 23), ('1348527119601013141512', 'ruullddrrd', 11), ('1234568110107129131415', 'drrruuldrd', 14)]

#4x4 40 Steps
['l', 'u', 'l', 'u', 'l', 'd', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 6, 10, 7, 8, 5, 11, 14, 12, 9, 13, 15, 0]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
['l', 'u', 'u', 'u', 'r', 'r', 'd', 'r', 'd', 'd']
-----------------------
[5, 1, 2, 4, 9, 6, 3, 7, 13, 10, 11, 8, 14, 0, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['l', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 6, 7, 8, 5, 9, 15, 11, 13, 10, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1234610785111412913150', 'lululddrrr', 11), ('5124963713101181401512', 'luuurrdrdd', 10), ('2304167859151113101412', 'llddrdrurd', 10)]

#4x4 80 Steps
['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 21
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 16
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 23
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 21), ('2534101179610813141512', 'uldrdrurdd', 16), ('1273564891011121301415', 'rruulurddd', 23)]

### AStar using OutOfPlace w/ Test Problems

In [154]:
AStarFb = lambda n: n.PathCost + OutOfPlace(n.State, GoalState)

Solutions = []
for s in problemList:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, AStarFb)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 47
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 23
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 40
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 47), ('2534101179610813141512', 'uldrdrurdd', 23), ('1273564891011121301415', 'rruulurddd', 40)]


#Results:
#4x4 5 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'u', 'u', 'l', 'd', 'r', 'd', 'l', 'd', 'r']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 51
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 10), ('1247563891001213141115', 'ruuldrdldr', 51), ('2304110685971113141512', 'llddrurdrd', 10)]

#4x4 10 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'u', 'u', 'l', 'd', 'r', 'd', 'l', 'd', 'r']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 51
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 10), ('1247563891001213141115', 'ruuldrdldr', 51), ('2304110685971113141512', 'llddrurdrd', 10)]

#4x4 20 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'u', 'u', 'l', 'd', 'r', 'd', 'l', 'd', 'r']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 51
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 10), ('1247563891001213141115', 'ruuldrdldr', 51), ('2304110685971113141512', 'llddrurdrd', 10)]

#4x4 40 Steps
['l', 'u', 'l', 'u', 'l', 'd', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 6, 10, 7, 8, 5, 11, 14, 12, 9, 13, 15, 0]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 19
-----------------------
['l', 'u', 'u', 'u', 'r', 'r', 'd', 'r', 'd', 'd']
-----------------------
[5, 1, 2, 4, 9, 6, 3, 7, 13, 10, 11, 8, 14, 0, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['l', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 6, 7, 8, 5, 9, 15, 11, 13, 10, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1234610785111412913150', 'lululddrrr', 19), ('5124963713101181401512', 'luuurrdrdd', 10), ('2304167859151113101412', 'llddrdrurd', 10)]

#4x4 80 Steps
['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 47
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 23
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 40
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 47), ('2534101179610813141512', 'uldrdrurdd', 23), ('1273564891011121301415', 'rruulurddd', 40)]

### Best First Search -- Greedy

In [174]:
### Best First
bestFirstSearchf = lambda n: OutOfPlace(n.State, GoalState)

Solutions = []
for s in problemList:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, bestFirstSearchf)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'r', 'd', 'r', 'u', 'l', 'd', 'r', 'd', 'd']
-----------------------
[0, 1, 2, 7, 5, 6, 4, 3, 9, 10, 11, 8, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 18
-----------------------
['d', 'l', 'u', 'l', 'd', 'r', 'd', 'r', 'r', 'd']
-----------------------
[2, 5, 0, 4, 1, 7, 3, 8, 9, 6, 10, 11, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 33
-----------------------
['r', 'r', 'd', 'd', 'l', 'l', 'u', 'l', 'u', 'r', 'd', 'd', 'l', 'u', 'r', 'u', 'l', 'd', 'd', 'r', 'r', 'u', 'l', 'd', 'l', 'u', 'r', 'r', 'd', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 9, 0, 6, 7, 13, 5, 11, 8, 14, 10, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  36
Nodes Expanded= 401
-----------------------
-------
[('0127564391011813141512', 'rrdruldrdd', 18), ('2504173896101113141512', 'dluldrdrrd', 33), ('1234

#Results
#4x4 5 Steps
['r', 'r', 'd', 'r', 'u', 'l', 'd', 'r', 'd', 'd']
-----------------------
[0, 1, 2, 7, 5, 6, 4, 3, 9, 10, 11, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 18
-----------------------
['d', 'l', 'u', 'l', 'd', 'r', 'd', 'r', 'r', 'd']
-----------------------
[2, 5, 0, 4, 1, 7, 3, 8, 9, 6, 10, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 33
-----------------------
['r', 'r', 'd', 'd', 'l', 'l', 'u', 'l', 'u', 'r', 'd', 'd', 'l', 'u', 'r', 'u', 'l', 'd', 'd', 'r', 'r', 'u', 'l', 'd', 'l', 'u', 'r', 'r', 'd', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 9, 0, 6, 7, 13, 5, 11, 8, 14, 10, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  36
Nodes Expanded= 401
-----------------------
-------
[('0127564391011813141512', 'rrdruldrdd', 18), ('2504173896101113141512', 'dluldrdrrd', 33), ('1234906713511814101512', 'rrddllulurddlurulddrruldlurrdluldrrr', 401)]

#4x4 10 Steps
['d', 'l', 'l', 'l', 'd', 'r', 'u', 'r', 'r', 'd', 'l', 'u', 'l', 'd', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 11, 14, 8, 9, 13, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  16
Nodes Expanded= 35
-----------------------
['r', 'd', 'd', 'l', 'u', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 0, 7, 5, 6, 11, 3, 9, 10, 8, 4, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 140
-----------------------
['d', 'd', 'r', 'u', 'r', 'd', 'l', 'l', 'l', 'u', 'r', 'd', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 0, 7, 8, 13, 6, 15, 11, 10, 9, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  14
Nodes Expanded= 20
-----------------------
-------
[('1234567010111489131512', 'dllldrurrdluldrr', 35), ('1207561139108413141512', 'rddluurddd', 140), ('1234507813615111091412', 'ddrurdlllurdrr', 20)]

#4x4 20 Steps
['d', 'd', 'd', 'r', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[0, 2, 3, 4, 1, 6, 7, 8, 5, 13, 11, 12, 10, 9, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 18
-----------------------
['d', 'd', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 0, 8, 11, 9, 6, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 12
-----------------------
['l', 'd', 'd', 'l', 'd', 'r', 'u', 'r', 'r', 'd']
-----------------------
[1, 3, 0, 4, 5, 2, 7, 8, 14, 6, 10, 11, 9, 13, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 113
-----------------------
-------
[('0234167851311121091415', 'dddruldrrr', 18), ('1234508119671213101415', 'ddrruuldrd', 12), ('1304527814610119131512', 'lddldrurrd', 113)]

#4x4 40 Steps
['l', 'd', 'l', 'd', 'd', 'r', 'r', 'u', 'r', 'd']
-----------------------
[1, 3, 0, 4, 6, 2, 7, 8, 5, 10, 15, 11, 9, 13, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'u', 'u', 'r', 'd', 'r', 'd', 'l', 'd', 'r']
-----------------------
[1, 6, 2, 4, 5, 10, 3, 7, 0, 9, 12, 8, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'u', 'r', 'u', 'l', 'l', 'd', 'r', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 7, 8, 11, 9, 6, 15, 10, 13, 0, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 352
-----------------------
-------
[('1304627851015119131412', 'ldlddrrurd', 10), ('1624510370912813141115', 'ruurdrdldr', 10), ('1234578119615101301412', 'rurulldrrd', 352)]

#4x4 80 Steps
['r', 'r', 'u', 'l', 'd', 'r', 'u', 'l', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  12
Nodes Expanded= 50
-----------------------
['d', 'r', 'u', 'l', 'u', 'l', 'd', 'r', 'r', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  12
Nodes Expanded= 128
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 16
-----------------------
-------
[('1234561179101215130148', 'rruldrulurdd', 50), ('2534101179610813141512', 'drululdrrrdd', 128), ('1273564891011121301415', 'rruulurddd', 16)]

In [156]:
### Best First
bestFirstSearchf = lambda n: ManhattanDistance(n.State, GoalState)

Solutions = []
for s in problemList:
  TileSliding.INITIAL = s
  ret, cost = BestFirstSearch(TileSliding, bestFirstSearchf)
  sol = Solution(ret)
  print (sol)
  print ("-----------------------")
  print (TileSliding.INITIAL,'\n')
  print (ApplyMoves(sol, TileSliding.INITIAL))
  print ("Length of solution: ", len(sol))
  print ("Nodes Expanded=", cost)
  print ("-----------------------")
  Solutions.append((''.join(map(str, s)), ''.join(sol), cost))
print ("-------")
print (Solutions)

['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 12
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15] 

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 12), ('2534101179610813141512', 'uldrdrurdd', 10), ('1273564891011121301415', 'rruulurddd', 11)]


#Results:
#4x4 5 Steps
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
['r', 'd', 'd', 'd', 'l', 'l', 'u', 'r', 'r', 'd']
-----------------------
[1, 2, 0, 3, 5, 6, 7, 4, 9, 14, 10, 8, 13, 15, 12, 11]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['u', 'r', 'r', 'd', 'd', 'l', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'l', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[1, 6, 2, 3, 5, 0, 11, 4, 9, 7, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  20
Nodes Expanded= 201
-----------------------
-------
[('1273564891011121301415', 'rruulurddd', 11), ('1203567491410813151211', 'rdddllurrd', 10), ('1623501149710813141512', 'urrddllurdrulldrurdd', 201)]

#4x4 10 Steps
['u', 'r', 'r', 'd', 'l', 'd', 'l', 'd', 'r', 'r']
-----------------------
[1, 6, 2, 3, 5, 0, 8, 4, 9, 11, 7, 12, 13, 10, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['d', 'r', 'u', 'u', 'u', 'l', 'd', 'r', 'd', 'd']
-----------------------
[1, 2, 4, 7, 5, 6, 3, 8, 9, 10, 0, 12, 13, 14, 11, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 12
-----------------------
['l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 10, 6, 8, 5, 9, 7, 11, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1623508491171213101415', 'urrdldldrr', 10), ('1247563891001213141115', 'druuuldrdd', 12), ('2304110685971113141512', 'llddrurdrd', 10)]

#4x4 20 Steps
['d', 'd', 'l', 'l', 'u', 'l', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 5, 6, 7, 0, 10, 13, 11, 8, 9, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 12
-----------------------
['r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'd']
-----------------------
[1, 3, 4, 8, 5, 2, 7, 11, 9, 6, 0, 10, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['d', 'r', 'r', 'r', 'u', 'u', 'l', 'd', 'r', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 8, 11, 0, 10, 7, 12, 9, 13, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1234567010131189141512', 'ddlluldrrr', 12), ('1348527119601013141512', 'ruullddrrd', 10), ('1234568110107129131415', 'drrruuldrd', 10)]

#4x4 40 Steps
['l', 'u', 'l', 'u', 'l', 'd', 'd', 'r', 'r', 'r']
-----------------------
[1, 2, 3, 4, 6, 10, 7, 8, 5, 11, 14, 12, 9, 13, 15, 0]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
['l', 'u', 'u', 'u', 'r', 'r', 'd', 'r', 'd', 'd']
-----------------------
[5, 1, 2, 4, 9, 6, 3, 7, 13, 10, 11, 8, 14, 0, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['l', 'l', 'd', 'd', 'r', 'd', 'r', 'u', 'r', 'd']
-----------------------
[2, 3, 0, 4, 1, 6, 7, 8, 5, 9, 15, 11, 13, 10, 14, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
-------
[('1234610785111412913150', 'lululddrrr', 11), ('5124963713101181401512', 'luuurrdrdd', 10), ('2304167859151113101412', 'llddrdrurd', 10)]

#4x4 80 Steps
['r', 'u', 'r', 'd', 'l', 'u', 'u', 'r', 'd', 'd']
-----------------------
[1, 2, 3, 4, 5, 6, 11, 7, 9, 10, 12, 15, 13, 0, 14, 8]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 12
-----------------------
['u', 'l', 'd', 'r', 'd', 'r', 'u', 'r', 'd', 'd']
-----------------------
[2, 5, 3, 4, 1, 0, 11, 7, 9, 6, 10, 8, 13, 14, 15, 12]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 10
-----------------------
['r', 'r', 'u', 'u', 'l', 'u', 'r', 'd', 'd', 'd']
-----------------------
[1, 2, 7, 3, 5, 6, 4, 8, 9, 10, 11, 12, 13, 0, 14, 15]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0]
Length of solution:  10
Nodes Expanded= 11
-----------------------
-------
[('1234561179101215130148', 'rurdluurdd', 12), ('2534101179610813141512', 'uldrdrurdd', 10), ('1273564891011121301415', 'rruulurddd', 11)]

# Domain 2

In [175]:
VectorWorldDim = 10
VectorWorld = Problem()
VectorWorld.INITIAL = [0]
VectorWorld.IsGoal = lambda s: s==[3,] or s==(3,)
VectorWorld.Actions = lambda s: ['Left', 'Right']
## TileSliding.Result=VectorWorldResult
VectorWorld.ActionCost = lambda s, a, sPrime: 1

In [176]:
def VectorWorldResult(state, action):
  if action=='Left':
    return [(state[0]+VectorWorldDim-1)%VectorWorldDim]
  else:
    return [(state[0]+1)%VectorWorldDim]
VectorWorld.Result=VectorWorldResult


In [177]:
print (VectorWorld.IsGoal((3,)))

True


In [178]:
ret, cost = BreadthFirstSearch(VectorWorld)
print ("ret=", ret)
sol = Solution(ret)
print (sol)


ret= [3], Right
['Right', 'Right', 'Right']


In [179]:
VectorWorld.INITIAL = [8]

In [180]:
ret, cost = BreadthFirstSearch(VectorWorld)
print ("ret=", ret)
sol = Solution(ret)
print (sol)

ret= [3], Left
['Left', 'Left', 'Left', 'Left', 'Left']


#Explanation of Results
For the following with 4x4 problems:

* Breadth First Search generally had many nodes expanded for every problem, with a slight decrease when increasing the step amount. Overall, this searching algorithm was consistently slow in searching, but did show improvements with more steps.

* Uniform Cost Search showed incredibly slow performance, searching upwards of 5000 nodes, only slightly decreasing with an increase of steps.

* AStar using ManhattanDistance showed to have very little nodes expanded (10-20) for every step increment. There seemed to be a decline in performance when increasing step count, but this could be because of a lack of test cases.

* AStar using OutOfPlace demonstrated similar performance for lower step counts, but fluctuated from 10 to 50. With 80 steps, more nodes were expanded for every problem.

* Best First Search (Greedy) demonstrated the most inconsistent performance with some finding solutions within 10 nodes expanded, and other times, with a few hundred nodes expanded.

* Best First (with Manhattan Distance) demonstrated the best performance of all the searching algorithms, with the most consistent least nodes searched for all step increments. At 5 steps, it showed some fluctuation from 10 to several hundred nodes, similarly to the Greedy algorithm. However, at 10 and beyond steps, it showed to be very consistent in low node counts.


These findings highlight a fundamental principle in artificial intelligence: heuristic-driven search is crucial for solving complex problems efficiently. As the size and complexity of a problem increase, uninformed strategies like Breadth-First Search become computationally impractical due to the exponential growth of the search space. In contrast, informed algorithms such as A* or Greedy Best-First Search remain viable because they use heuristics to guide the search more intelligently—provided those heuristics are well-designed and appropriate for the problem domain. This insight is highly applicable to real-world AI challenges, where the complexity often resembles that of larger puzzles like the 4x4 version. Ultimately, the effectiveness and scalability of an AI system depend significantly on selecting the right search algorithm and developing heuristics tailored to the specific task.